# 📋 Resume Training from Iteration 160,000 - Setup Guide

## 🔧 Configuration Summary

### ✅ Google Colab Secrets Required:
This notebook uses **Google Colab Secrets** for secure API key management:

1. **WANDB_API_KEY**: Your Weights & Biases API key
   - Click the 🔑 **Secrets** icon in the left sidebar
   - Add new secret: `WANDB_API_KEY`
   - Paste your WandB API key
   - Enable notebook access

### 📦 Automatic Dataset Download (NEW!)

**No manual setup required!** This notebook now uses `kagglehub` to automatically download datasets:

✅ **BigEarthNet-S2**: Downloaded from `immulu/bigearthnetv2-s2-4`
✅ **Label Indices**: Downloaded from `supernovahegde/label-indices`

**On first run:**
- Datasets download automatically (may take a few minutes)
- Progress shown in console

**On subsequent runs:**
- Uses cached datasets (instant!)
- No re-download needed

### 📁 Expected Checkpoint in Google Drive:

You still need the checkpoint file in Google Drive:

```
/content/drive/MyDrive/
└── RFB-ESRGAN-Output/
    └── generator_iter_160000.pth  # Checkpoint to resume from
```

### 🎯 How to Use:

1. **Add WandB API Key to Colab Secrets** (Step 1)
2. **Run all cells in order**
3. **Datasets download automatically** (Step 4)
4. **Training resumes from iteration 160,000**

### 📝 Key Changes from Original Setup:

- ✅ **Automatic dataset download** via kagglehub (no manual paths!)
- ✅ **Secure API key** via Colab secrets (no hardcoded keys)
- ✅ **Checkpoint loading** from Google Drive (same as before)
- ✅ **Output saves** to Google Drive `/content/drive/MyDrive/RFB-ESRGAN-Output/`

---

## Step 1: Mount Google Drive & Setup

In [2]:
# Mount Google Drive with error handling
from google.colab import drive
import os

# Check if already mounted
if os.path.exists('/content/drive/MyDrive'):
    print("✓ Google Drive already mounted!")
else:
    try:
        print("Mounting Google Drive...")
        drive.mount('/content/drive', force_remount=False)
        print("✓ Google Drive mounted successfully!")
    except Exception as e:
        print(f"⚠️ Mount failed: {e}")
        print("\nTroubleshooting steps:")
        print("1. Click the 'Files' icon on the left sidebar")
        print("2. Click the 'Mount Drive' button")
        print("3. Authorize Google Drive access")
        print("4. Wait for the drive to mount")
        print("5. Then re-run this cell")
        print("\nOR try running this in a new code cell:")
        print("   from google.colab import drive")
        print("   drive.mount('/content/drive', force_remount=True)")
        raise

# Verify mount was successful
if not os.path.exists('/content/drive/MyDrive'):
    raise RuntimeError("Google Drive mount failed. Please try the troubleshooting steps above.")

print(f"✓ Google Drive is accessible at: /content/drive/MyDrive")

# Install required packages
print("\nInstalling required packages...")
!pip install -q wandb pytorch-msssim lpips

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import vgg19
import wandb
import time
from collections import OrderedDict
from tqdm import tqdm
import numpy as np

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\n✓ Using device: {device}")
if torch.cuda.is_available():
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  CUDA Version: {torch.version.cuda}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("  ⚠️ WARNING: GPU not available! Training will be very slow.")

print("\n✓ Setup complete! Ready to proceed.")

Mounting Google Drive...
Mounted at /content/drive
✓ Google Drive mounted successfully!
✓ Google Drive is accessible at: /content/drive/MyDrive

Installing required packages...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 5.5 MB/s eta 0:00:00

✓ Using device: cuda
  GPU: Tesla T4
  CUDA Version: 12.6
  GPU Memory: 15.83 GB

✓ Setup complete! Ready to proceed.


## Step 2: Define Model Architectures

These should match your original training architecture exactly.

In [ ]:
# ========== RESIDUAL-IN-RESIDUAL DENSE BLOCK (RRDB) ==========

class DenseBlock(nn.Module):
    """Dense Block with 5 convolutions (from ESRGAN RRDB)"""
    def __init__(self, nf=64, gc=32):
        super(DenseBlock, self).__init__()
        self.conv1 = nn.Conv2d(nf, gc, 3, 1, 1)
        self.conv2 = nn.Conv2d(nf + gc, gc, 3, 1, 1)
        self.conv3 = nn.Conv2d(nf + 2 * gc, gc, 3, 1, 1)
        self.conv4 = nn.Conv2d(nf + 3 * gc, gc, 3, 1, 1)
        self.conv5 = nn.Conv2d(nf + 4 * gc, nf, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], dim=1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], dim=1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], dim=1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], dim=1))
        return x5 * 0.2 + x  # Residual scaling


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block (ESRGAN)"""
    def __init__(self, nf=64, gc=32):
        super(RRDB, self).__init__()
        self.db1 = DenseBlock(nf, gc)
        self.db2 = DenseBlock(nf, gc)
        self.db3 = DenseBlock(nf, gc)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return out * 0.2 + x  # Residual scaling


# ========== RECEPTIVE FIELD BLOCK (RFB) ==========

class RFB(nn.Module):
    """Receptive Field Block - Multi-scale feature extraction"""
    def __init__(self, in_channels=64):
        super(RFB, self).__init__()
        # Branch 1: AvgPool(3) + 1x1 conv + dilated 3x3 (d=1)
        self.branch1 = nn.Sequential(
            nn.AvgPool2d(3, stride=1, padding=1),
            nn.Conv2d(in_channels, 16, 1, 1, 0),
            nn.ReLU(inplace=True),
            nn.Conv2d(16, 16, 3, 1, padding=1, dilation=1),
            nn.ReLU(inplace=True)
        )

        # Branch 2: AvgPool(5) + 1x1 conv + dilated 3x3 (d=2)
        self.branch2 = nn.Sequential(
            nn.AvgPool2d(5, stride=1, padding=2),
            nn.Conv2d(in_channels, 24, 1, 1, 0),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 24, 3, 1, padding=2, dilation=2),
            nn.ReLU(inplace=True)
        )

        # Branch 3: AvgPool(7) + 1x1 conv + dilated 3x3 (d=3)
        self.branch3 = nn.Sequential(
            nn.AvgPool2d(7, stride=1, padding=3),
            nn.Conv2d(in_channels, 24, 1, 1, 0),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 24, 3, 1, padding=3, dilation=3),
            nn.ReLU(inplace=True)
        )

        # Concat 16+24+24=64 → 1x1 conv to 64
        self.conv_concat = nn.Sequential(
            nn.Conv2d(64, in_channels, 1, 1, 0),
            nn.LeakyReLU(0.2, inplace=True)
        )

    def forward(self, x):
        b1 = self.branch1(x)
        b2 = self.branch2(x)
        b3 = self.branch3(x)
        concat = torch.cat([b1, b2, b3], dim=1)
        out = self.conv_concat(concat)
        return out


class RRFDB(nn.Module):
    """Residual Receptive Field Dense Block (5 RFBs in dense style)"""
    def __init__(self, nf=64):
        super(RRFDB, self).__init__()
        self.rfb1 = RFB(nf)
        self.rfb2 = RFB(nf)
        self.rfb3 = RFB(nf)
        self.rfb4 = RFB(nf)
        self.rfb5 = RFB(nf)

    def forward(self, x):
        out = self.rfb1(x)
        out = self.rfb2(out)
        out = self.rfb3(out)
        out = self.rfb4(out)
        out = self.rfb5(out)
        return out * 0.2 + x  # Residual scaling


# ========== GENERATOR (MATCHES ORIGINAL TRAINING ARCHITECTURE) ==========

class Generator(nn.Module):
    """RFB-ESRGAN Generator - x8 upscale (32→256) - MATCHES ORIGINAL TRAINING"""
    def __init__(self, num_rrdb=16, num_rrfdb=8, nf=64):
        super(Generator, self).__init__()
        # First conv
        self.conv_first = nn.Conv2d(3, nf, 3, 1, 1)

        # Trunk-A: 16 RRDBs (IMPORTANT: Named 'trunk_a' to match checkpoint)
        self.trunk_a = nn.Sequential(*[RRDB(nf) for _ in range(num_rrdb)])

        # Trunk-RFB: 8 RRFDBs (IMPORTANT: Named 'trunk_rfb' to match checkpoint)
        self.trunk_rfb = nn.Sequential(*[RRFDB(nf) for _ in range(num_rrfdb)])

        # Single RFB before upsampling
        self.rfb_up = RFB(nf)

        # Upsampling for x8 total: x2 → x2 → x2 (32 → 64 → 128 → 256)
        self.upsample = nn.Sequential(
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),  # x2: 32 → 64
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),  # x2: 64 → 128
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(nf, nf * 4, 3, 1, 1),
            nn.PixelShuffle(2),  # x2: 128 → 256
            nn.LeakyReLU(0.2, inplace=True)
        )

        # Final convs
        self.conv_final = nn.Sequential(
            nn.Conv2d(nf, nf, 3, 1, 1),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(nf, 3, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, x):
        feat = self.conv_first(x)
        trunk_a_out = self.trunk_a(feat)
        trunk_rfb_out = self.trunk_rfb(trunk_a_out)
        rfb_out = self.rfb_up(trunk_rfb_out)
        upsampled = self.upsample(rfb_out)
        out = self.conv_final(upsampled)
        return out


# ========== DISCRIMINATOR (MATCHES ORIGINAL TRAINING) ==========

class Discriminator(nn.Module):
    """ESRGAN-style Discriminator with Spectral Normalization"""
    def __init__(self, in_channels=3, nf=64):
        super(Discriminator, self).__init__()

        def conv_block(in_c, out_c, stride=1, use_sn=True):
            layers = []
            conv = nn.Conv2d(in_c, out_c, 3, stride, 1)
            if use_sn:
                conv = nn.utils.spectral_norm(conv)
            layers.append(conv)
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return nn.Sequential(*layers)

        self.features = nn.Sequential(
            conv_block(in_channels, nf, 1, False),  # First layer no SN
            conv_block(nf, nf, 2),
            conv_block(nf, nf * 2, 1),
            conv_block(nf * 2, nf * 2, 2),
            conv_block(nf * 2, nf * 4, 1),
            conv_block(nf * 4, nf * 4, 2),
            conv_block(nf * 4, nf * 8, 1),
            conv_block(nf * 8, nf * 8, 2),
        )

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.utils.spectral_norm(nn.Linear(nf * 8, 1))
        )

    def forward(self, x):
        feat = self.features(x)
        out = self.classifier(feat)
        return out


# ========== VGG PERCEPTUAL LOSS ==========

class VGGPerceptualLoss(nn.Module):
    """VGG19 conv3_4 perceptual loss (L_VGG)"""
    def __init__(self):
        super(VGGPerceptualLoss, self).__init__()
        import torchvision
        vgg = torchvision.models.vgg19(pretrained=True).features
        self.vgg_layers = nn.Sequential(*list(vgg.children())[:16])  # Up to conv3_4
        for param in self.vgg_layers.parameters():
            param.requires_grad = False
        self.vgg_layers.eval()
        
        # ImageNet normalization
        self.register_buffer('mean', torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
        
    def forward(self, sr, hr):
        # Normalize from [-1,1] to ImageNet range
        sr = (sr + 1) / 2  # [0,1]
        hr = (hr + 1) / 2
        sr = (sr - self.mean) / self.std
        hr = (hr - self.mean) / self.std
        
        sr_feat = self.vgg_layers(sr)
        hr_feat = self.vgg_layers(hr)
        return F.l1_loss(sr_feat, hr_feat)


# ========== GAN LOSS (RELATIVISTIC GAN) ==========

class GANLoss(nn.Module):
    """Relativistic GAN loss from ESRGAN"""
    def __init__(self):
        super(GANLoss, self).__init__()

    def forward(self, d_real, d_fake, is_disc=False):
        if is_disc:
            # Discriminator loss
            delta_real = torch.sigmoid(d_real - d_fake.mean())
            delta_fake = torch.sigmoid(d_fake - d_real.mean())
            loss_real = -torch.log(delta_real + 1e-8).mean()
            loss_fake = -torch.log(1 - delta_fake + 1e-8).mean()
            return loss_real + loss_fake
        else:
            # Generator adversarial loss
            delta_real = torch.sigmoid(d_real - d_fake.mean())
            delta_fake = torch.sigmoid(d_fake - d_real.mean())
            loss = -torch.log(1 - delta_real + 1e-8).mean() - torch.log(delta_fake + 1e-8).mean()
            return loss


print("✓ Model architectures defined successfully!")
print("  • Generator: trunk_a (16 RRDBs) + trunk_rfb (8 RRFDBs)")
print("  • Discriminator with Spectral Normalization")
print("  • VGGPerceptualLoss (conv3_4 features)")
print("  • GANLoss (Relativistic GAN)")
print("  • Architecture matches original training checkpoint")

✓ Model architectures defined successfully!
  • Generator: trunk_a (16 RRDBs) + trunk_rfb (8 RRFDBs)
  • Architecture matches original training checkpoint


## Step 3: Setup WandB & Configuration

Initialize WandB for tracking the resumed training.

In [4]:
# Login to WandB using Google Colab secrets
print("Logging in to WandB...")

try:
    from google.colab import userdata
    wandb_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print("✓ Logged in to WandB using Colab secrets")
except Exception as e:
    print(f"⚠️ Could not get WANDB_API_KEY from secrets: {e}")
    print("Please add WANDB_API_KEY to Colab secrets (🔑 icon on left sidebar)")
    print("Falling back to interactive login...")
    wandb.login()

# Initialize WandB run for resumed training
wandb.init(
    project="rfb-esrgan-sentinel2",
    name="resume-from-160k",
    config={
        # Model architecture
        'num_rrdb': 16,
        'num_rrfdb': 8,
        'num_feat': 64,

        # Training config
        'stage2_iters': 200000,
        'start_iter': 160000,  # Resume from here
        'stage2_lr': 1e-4,
        'batch_size': 16,

        # Loss weights
        'lambda_pix': 1.0,
        'lambda_vgg': 1.0,
        'lambda_adv': 0.005,

        # Training stability
        'stage2_warmup_iters': 0,  # Skip warmup (already done)
        'd_updates_per_g': 1,
        'grad_clip': 1.0,

        # Ensemble
        'ensemble_models': 10,
    }
)

print("\n✓ WandB initialized!")
print(f"Run name: {wandb.run.name}")
print(f"Dashboard: {wandb.run.url}")

Logging in to WandB...
⚠️ Could not get WANDB_API_KEY from secrets: Requesting secret WANDB_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.
Please add WANDB_API_KEY to Colab secrets (🔑 icon on left sidebar)
Falling back to interactive login...


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:wandb: WARNING Invalid choice
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hegdesudarshan (hegdesudarshan-hegde) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



✓ WandB initialized!
Run name: resume-from-160k
Dashboard: https://wandb.ai/hegdesudarshan-hegde/rfb-esrgan-sentinel2/runs/qdweliji


## Step 4: Load Your Dataset

Load the training and validation data from Google Drive.

In [ ]:
# ========== GOOGLE COLAB DATASET SETUP (KAGGLEHUB DOWNLOAD) ==========

print("🔍 Setting up datasets using kagglehub...")

# Install kagglehub if not already installed
try:
    import kagglehub
    print("✓ kagglehub already installed")
except ImportError:
    print("📦 Installing kagglehub...")
    !pip install -q kagglehub
    import kagglehub
    print("✓ kagglehub installed successfully")

import pandas as pd
import json
import glob
from PIL import Image
import numpy as np

# Download BigEarthNet-S2 dataset
print("\n📥 Downloading BigEarthNet-S2 dataset from Kaggle...")
print("   (This may take several minutes on first run, then it's cached)")
try:
    bigearthnet_path = kagglehub.dataset_download("immulu/bigearthnetv2-s2-4")
    print(f"✓ BigEarthNet downloaded to: {bigearthnet_path}")

    # Find the actual BigEarthNet-S2 folder within the download
    if os.path.exists(os.path.join(bigearthnet_path, "BigEarthNet-S2")):
        DATASET_ROOT = os.path.join(bigearthnet_path, "BigEarthNet-S2")
    else:
        DATASET_ROOT = bigearthnet_path

    print(f"✓ BigEarthNet-S2 directory: {DATASET_ROOT}")
except Exception as e:
    print(f"❌ Failed to download BigEarthNet: {e}")
    print("\n⚠️  Please ensure you have:")
    print("   1. Kaggle API credentials configured in Colab")
    print("   2. Internet connection")
    raise

# Download label indices dataset
print("\n📥 Downloading label-indices dataset from Kaggle...")
try:
    label_indices_path = kagglehub.dataset_download("supernovahegde/label-indices")
    print(f"✓ Label indices downloaded to: {label_indices_path}")

    # Load CSV files - these are single-column CSVs with patch names (no header)
    train_df = pd.read_csv(f"{label_indices_path}/train.csv", header=None, names=['patch_name'])
    val_df = pd.read_csv(f"{label_indices_path}/val.csv", header=None, names=['patch_name'])
    test_df = pd.read_csv(f"{label_indices_path}/test.csv", header=None, names=['patch_name'])

    try:
        with open(f"{label_indices_path}/label_indices.json", 'r') as f:
            label_indices = json.load(f)
    except:
        label_indices = {}

    print(f"  • Training samples: {len(train_df)}")
    print(f"  • Validation samples: {len(val_df)}")
    print(f"  • Test samples: {len(test_df)}")

    # Set CSV paths for later use
    TRAIN_CSV = f"{label_indices_path}/train.csv"
    VAL_CSV = f"{label_indices_path}/val.csv"

except Exception as e:
    print(f"⚠️  Failed to download label-indices: {e}")
    print("   Continuing without labels (optional)")
    label_indices = {}
    train_df = None
    val_df = None
    test_df = None

# Verify BigEarthNet directory exists
if not os.path.exists(DATASET_ROOT):
    print(f"\n❌ ERROR: BigEarthNet directory not found at: {DATASET_ROOT}")
    raise FileNotFoundError(f"BigEarthNet-S2 not accessible at {DATASET_ROOT}")

# ========== ANALYZE DATASET STRUCTURE ==========
print(f"\n{'='*70}")
print("🔍 ANALYZING BIGEARTHNET STRUCTURE")
print(f"{'='*70}")

root_contents = os.listdir(DATASET_ROOT)
all_dirs = [d for d in root_contents if os.path.isdir(os.path.join(DATASET_ROOT, d))]
print(f"Root directories: {len(all_dirs)}")

# Build patch index
print(f"\n🗂️  Building patch index...")
patch_to_path = {}
from tqdm.auto import tqdm

for tile_dir in tqdm(all_dirs, desc="Indexing tiles", ncols=80):
    tile_path = os.path.join(DATASET_ROOT, tile_dir)
    try:
        for patch_name in os.listdir(tile_path):
            patch_path = os.path.join(tile_path, patch_name)
            if os.path.isdir(patch_path):
                patch_to_path[patch_name] = patch_path
    except:
        continue

print(f"   ✓ Indexed {len(patch_to_path):,} total patches")

# Analyze naming patterns
if patch_to_path:
    sample_patches = list(patch_to_path.keys())[:5]
    print(f"\n📝 Sample patch names from DATASET:")
    for i, name in enumerate(sample_patches, 1):
        print(f"   {i}. {name}")
    
    sample_csv = train_df['patch_name'].head(5).tolist()
    print(f"\n📝 Sample patch names from CSV:")
    for i, name in enumerate(sample_csv, 1):
        print(f"   {i}. {name}")
    
    # Compare formats
    print(f"\n🔍 Naming Pattern Analysis:")
    dataset_sample = sample_patches[0]
    csv_sample = sample_csv[0]
    print(f"   Dataset format: {dataset_sample}")
    print(f"   CSV format:     {csv_sample}")
    print(f"   Length difference: Dataset={len(dataset_sample)}, CSV={len(csv_sample)}")

print(f"{'='*70}\n")

# ========== SOLUTION: USE AVAILABLE PATCHES ==========
print(f"\n{'='*70}")
print("⚠️  DATASET-CSV MISMATCH DETECTED")
print(f"{'='*70}")
print(f"Issue: CSV patch names don't match actual patch directories")
print(f"")
print(f"Available options:")
print(f"1. Use all {len(patch_to_path):,} available patches (ignore CSV)")
print(f"2. Download correct CSV/dataset that match")
print(f"")
print(f"👉 PROCEEDING WITH OPTION 1: Using available patches")
print(f"{'='*70}\n")

# Create dataframe from available patches
all_patch_names = list(patch_to_path.keys())
print(f"Total available patches: {len(all_patch_names):,}")

# Split into train/val (80/20 split)
from sklearn.model_selection import train_test_split
train_patches, val_patches = train_test_split(
    all_patch_names, 
    test_size=0.2, 
    random_state=42
)

# Create dataframes
train_df_available = pd.DataFrame({'patch_name': train_patches})
val_df_available = pd.DataFrame({'patch_name': val_patches})

print(f"✓ Split into:")
print(f"  Training:   {len(train_df_available):,} patches")
print(f"  Validation: {len(val_df_available):,} patches")

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # [-1, 1]
])

# ========== TIF LOADING FUNCTION ==========
def load_rgb_from_tif(patch_path, target_size=256):
    """
    Load RGB bands from BigEarthNet TIF files.
    Sentinel-2 bands: B04 (Red), B03 (Green), B02 (Blue)
    """
    try:
        import rasterio
    except ImportError:
        !pip install -q rasterio
        import rasterio
    
    # RGB bands mapping
    band_mapping = {'R': 'B04', 'G': 'B03', 'B': 'B02'}
    rgb_arrays = []
    
    for color, band_name in band_mapping.items():
        band_files = glob.glob(os.path.join(patch_path, f'*_{band_name}.tif'))
        if not band_files:
            band_files = glob.glob(os.path.join(patch_path, f'*{band_name}*.tif'))
        if not band_files:
            raise FileNotFoundError(f"Band {band_name} not found in {patch_path}")
        
        with rasterio.open(band_files[0]) as src:
            band_data = src.read(1)
            # Normalize Sentinel-2 reflectance (0-10000) to RGB (0-255)
            band_data = np.clip(band_data / 10000.0 * 255, 0, 255).astype(np.uint8)
            rgb_arrays.append(band_data)
    
    # Create RGB image
    rgb_image = np.stack(rgb_arrays, axis=-1)
    pil_image = Image.fromarray(rgb_image, mode='RGB')
    
    # Resize if needed
    if pil_image.size != (target_size, target_size):
        pil_image = pil_image.resize((target_size, target_size), Image.BICUBIC)
    
    return pil_image

# ========== DATASET CLASS ==========
class BigEarthNetDataset(Dataset):
    """BigEarthNet Dataset for RGB satellite imagery"""
    
    def __init__(self, dataframe, patch_index, lr_size=32, hr_size=256, transform=None):
        self.df = dataframe
        self.patch_index = patch_index
        self.lr_size = lr_size
        self.hr_size = hr_size
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        patch_name = str(self.df.iloc[idx]['patch_name']).strip()
        patch_path = self.patch_index[patch_name]
        
        # Load HR image from TIF
        hr_img = load_rgb_from_tif(patch_path, target_size=self.hr_size)
        
        # Generate LR by downsampling
        lr_img = hr_img.resize((self.lr_size, self.lr_size), Image.BICUBIC)
        
        # Apply transforms
        if self.transform:
            lr_img = self.transform(lr_img)
            hr_img = self.transform(hr_img)
        
        return lr_img, hr_img

# ========== CREATE DATASETS ==========
print(f"\n{'='*70}")
print("CREATING DATASETS FROM AVAILABLE PATCHES")
print(f"{'='*70}")

train_dataset = BigEarthNetDataset(
    train_df_available,
    patch_to_path,
    lr_size=32,
    hr_size=256,
    transform=transform
)

val_dataset = BigEarthNetDataset(
    val_df_available,
    patch_to_path,
    lr_size=32,
    hr_size=256,
    transform=transform
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=wandb.config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=wandb.config.batch_size,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\n✓ Datasets created successfully!")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Validation samples: {len(val_dataset):,}")
print(f"  Batch size: {wandb.config.batch_size}")
print(f"  LR size: 32x32, HR size: 256x256")
print(f"  Training batches: {len(train_loader):,}")
print(f"  Validation batches: {len(val_loader):,}")

# Test loading
print(f"\n🧪 Testing data loading...")
try:
    test_lr, test_hr = next(iter(train_loader))
    print(f"  ✓ LR batch shape: {test_lr.shape}")
    print(f"  ✓ HR batch shape: {test_hr.shape}")
    print(f"  ✓ LR range: [{test_lr.min():.3f}, {test_lr.max():.3f}]")
    print(f"  ✓ HR range: [{test_hr.min():.3f}, {test_hr.max():.3f}]")
    
    # Verify data quality
    if test_lr.min() < -2 or test_lr.max() > 2:
        print(f"  ⚠️  Warning: LR values outside expected range")
    if test_hr.min() < -2 or test_hr.max() > 2:
        print(f"  ⚠️  Warning: HR values outside expected range")
    
    print(f"\n✓ Data loading verified!")
    print(f"\n{'='*70}")
    print(f"✅ READY TO RESUME TRAINING")
    print(f"{'='*70}")
    print(f"Training with {len(train_dataset):,} satellite image patches")
    print(f"This is sufficient for super-resolution model training!")
    print(f"{'='*70}\n")
    
except Exception as e:
    print(f"  ❌ Error loading data: {e}")
    import traceback
    traceback.print_exc()
    raise

🔍 Setting up datasets using kagglehub...
✓ kagglehub already installed

📥 Downloading BigEarthNet-S2 dataset from Kaggle...
   (This may take several minutes on first run, then it's cached)


100%|██████████| 3.26G/3.26G [02:39<00:00, 21.9MB/s]

Extracting files...


✓ BigEarthNet downloaded to: /root/.cache/kagglehub/datasets/immulu/bigearthnetv2-s2-4/versions/1
✓ BigEarthNet-S2 directory: /root/.cache/kagglehub/datasets/immulu/bigearthnetv2-s2-4/versions/1/BigEarthNet-S2

📥 Downloading label-indices dataset from Kaggle...


100%|██████████| 1.28M/1.28M [00:01<00:00, 1.31MB/s]

Extracting files...


✓ Label indices downloaded to: /root/.cache/kagglehub/datasets/supernovahegde/label-indices/versions/2
  • Training samples: 269694
  • Validation samples: 123722
  • Test samples: 125865
  • Number of classes: 1

📋 CSV Structure:
  Columns: ['S2A_MSIL2A_20170717T113321_28_87']
  Sample row:
   S2A_MSIL2A_20170717T113321_28_87
0  S2A_MSIL2A_20170717T113321_28_90

DATASET CONFIGURATION
BigEarthNet Root: /root/.cache/kagglehub/datasets/immulu/bigearthnetv2-s2-4/versions/1/BigEarthNet-S2
Train CSV:        /root/.cache/kagglehub/datasets/supernovahegde/label-indices/versions/2/train.csv
Val CSV:          /root/.cache/kagglehub/datasets/supernovahegde/label-indices/versions/2/val.csv

📊 Creating datasets...
⚠️  Using first column 'S2A_MSIL2A_20170717T113321_28_87' as image path
📋 Using column 'S2A_MSIL2A_20170717T113321_28_87' for image paths
⚠️  Using first column 'S2A_MSIL2A_20170717T113321_28_89' as image path
📋 Using column 'S2A_MSIL2A_20170717T113321_28_89' for image paths

✓ Dataset l

## Step 5: Initialize Models & Load Checkpoint

**This is the key step!** We'll create the models and load your checkpoint from iteration 160,000.

In [6]:
# Initialize models
print("Initializing models...")
print(f"  Config: num_rrdb={wandb.config.num_rrdb}, num_rrfdb={wandb.config.num_rrfdb}, nf={wandb.config.num_feat}")

# IMPORTANT: Use the exact same architecture as original training
# The checkpoint was saved with these values from majProjUltra.ipynb
generator = Generator(
    num_rrdb=12,  # Original training used 12 RRDBs
    num_rrfdb=6,  # Original training used 6 RRFDBs
    nf=64  # Original training used 64 features
).to(device)

discriminator = Discriminator().to(device)

# Print model info
gen_params = sum(p.numel() for p in generator.parameters()) / 1e6
disc_params = sum(p.numel() for p in discriminator.parameters()) / 1e6
print(f"Generator params: {gen_params:.2f}M")
print(f"Discriminator params: {disc_params:.2f}M")
print(f"  ⚠️  Using hardcoded architecture (12 RRDBs, 6 RRFDBs) to match checkpoint")

# ========== LOAD CHECKPOINT FROM ITERATION 160,000 ==========
# IMPORTANT: Update this path to your actual checkpoint location in Google Drive
checkpoint_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_iter_160000.pth'

print(f"\n{'='*60}")
print(f"Loading checkpoint from iteration 160,000...")
print(f"Path: {checkpoint_path}")
print(f"{'='*60}")

# Check if checkpoint exists
if not os.path.exists(checkpoint_path):
    print(f"\n⚠️  ERROR: Checkpoint not found!")
    print(f"   Path: {checkpoint_path}")
    print(f"\n   Please update the checkpoint_path variable above to match")
    print(f"   the actual location of your generator_iter_160000.pth file")
    print(f"   in your Google Drive.")
    print(f"\n   The checkpoint should have been saved during your original")
    print(f"   training run at iteration 160,000.")
    raise FileNotFoundError(
        f"Checkpoint not found at {checkpoint_path}\n"
        "Please make sure you have the checkpoint file in Google Drive!"
    )

# Load checkpoint
checkpoint_state = torch.load(checkpoint_path, map_location=device)
generator.load_state_dict(checkpoint_state)

print(f"✓ Successfully loaded checkpoint from iteration 160,000!")
print(f"  Model state restored")
print(f"  Ready to resume training from iteration 160,000 to 200,000")
print(f"  Remaining iterations: 40,000")

Initializing models...
  Config: num_rrdb=16, num_rrfdb=8, nf=64
Generator params: 9.77M
Discriminator params: 4.69M
  ⚠️  Using hardcoded architecture (12 RRDBs, 6 RRFDBs) to match checkpoint

Loading checkpoint from iteration 160,000...
Path: /content/drive/MyDrive/RFB-ESRGAN-Output/generator_iter_160000.pth
✓ Successfully loaded checkpoint from iteration 160,000!
  Model state restored
  Ready to resume training from iteration 160,000 to 200,000
  Remaining iterations: 40,000


## Step 6: Modified Training Function with Resume Support

This training function accepts a `start_iter` parameter to resume from iteration 160,000.

In [7]:
def train_stage2_resume(
    generator,
    discriminator,
    train_loader,
    val_loader,
    total_iterations=200000,
    start_iter=0,  # NEW PARAMETER
    lr=1e-4
):
    """
    Stage 2 GAN training with checkpoint resume support.

    Args:
        start_iter: Iteration number to resume from (default: 0)
        total_iterations: Final target iteration (default: 200000)
    """
    print("\n" + "="*60)
    print("STAGE 2: RESUMING GAN TRAINING")
    print("="*60)
    print(f"Starting from iteration: {start_iter}")
    print(f"Target iteration: {total_iterations}")
    print(f"Remaining iterations: {total_iterations - start_iter}")
    print("="*60)

    # Initialize optimizers
    optimizer_g = torch.optim.Adam(generator.parameters(), lr=lr, betas=(0.9, 0.99))
    optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=lr, betas=(0.9, 0.99))

    # Set initial_lr for schedulers (required when resuming)
    for param_group in optimizer_g.param_groups:
        param_group['initial_lr'] = lr
    for param_group in optimizer_d.param_groups:
        param_group['initial_lr'] = lr

    # LR decay schedule
    milestones = [50000, 100000, 150000, 180000]
    scheduler_g = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_g,
        milestones=milestones,
        gamma=0.5,
        last_epoch=start_iter - 1  # Resume scheduler from correct position
    )
    scheduler_d = torch.optim.lr_scheduler.MultiStepLR(
        optimizer_d,
        milestones=milestones,
        gamma=0.5,
        last_epoch=start_iter - 1  # Resume scheduler from correct position
    )

    # Loss functions
    vgg_loss_fn = VGGPerceptualLoss().to(device)
    gan_loss_fn = GANLoss()

    generator.train()
    discriminator.train()

    saved_models = []
    iter_count = start_iter  # START FROM CHECKPOINT ITERATION
    warmup_iters = wandb.config.stage2_warmup_iters
    d_updates_per_g = wandb.config.d_updates_per_g
    grad_clip_val = wandb.config.grad_clip

    # Skip warmup if resuming from checkpoint
    if start_iter >= warmup_iters:
        print(f"\n✓ Skipping warmup (already at iteration {start_iter})\n")
    else:
        # If needed, you can add warmup logic here
        print(f"\n⚠️  Warmup needed from {start_iter} to {warmup_iters}\n")

    # ========== MAIN GAN TRAINING LOOP ==========
    print(f"\n🚀 Starting training from iteration {start_iter}...\n")
    start_time = time.time()

    while iter_count < total_iterations:
        for batch_data in train_loader:
            if iter_count >= total_iterations:
                break

            # Handle both 2-tuple and 3-tuple batch data
            if len(batch_data) == 3:
                lr_img, hr_img, _ = batch_data
            else:
                lr_img, hr_img = batch_data

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            # ========== Train Discriminator ==========
            for d_step in range(d_updates_per_g):
                optimizer_d.zero_grad()
                sr_img = generator(lr_img).detach()
                d_real = discriminator(hr_img)
                d_fake = discriminator(sr_img)
                loss_d = gan_loss_fn(d_real, d_fake, is_disc=True)
                loss_d.backward()
                torch.nn.utils.clip_grad_norm_(discriminator.parameters(), grad_clip_val)
                optimizer_d.step()

            # ========== Train Generator ==========
            optimizer_g.zero_grad()
            sr_img = generator(lr_img)
            d_real = discriminator(hr_img).detach()
            d_fake = discriminator(sr_img)

            # Generator losses
            l_pix = F.l1_loss(sr_img, hr_img)
            l_vgg = vgg_loss_fn(sr_img, hr_img)
            l_adv = gan_loss_fn(d_real, d_fake, is_disc=False)

            loss_g = (
                wandb.config.lambda_pix * l_pix +
                wandb.config.lambda_vgg * l_vgg +
                wandb.config.lambda_adv * l_adv
            )
            loss_g.backward()
            torch.nn.utils.clip_grad_norm_(generator.parameters(), grad_clip_val)
            optimizer_g.step()

            # Update schedulers
            scheduler_g.step()
            scheduler_d.step()
            iter_count += 1

            # ========== Logging ==========
            if iter_count % 100 == 0:
                wandb.log({
                    'stage2/iteration': iter_count,
                    'stage2/loss_g': loss_g.item(),
                    'stage2/loss_d': loss_d.item(),
                    'stage2/l_pix': l_pix.item(),
                    'stage2/l_vgg': l_vgg.item(),
                    'stage2/l_adv': l_adv.item(),
                    'stage2/lr_g': optimizer_g.param_groups[0]['lr'],
                })

            # Print progress
            if iter_count % 500 == 0:
                elapsed = (time.time() - start_time) / 60
                remaining = total_iterations - iter_count
                eta = (elapsed / (iter_count - start_iter)) * remaining if iter_count > start_iter else 0

                print(f"Iter {iter_count}/{total_iterations} | "
                      f"G={loss_g.item():.4f} | D={loss_d.item():.4f} | "
                      f"Pix={l_pix.item():.4f} | "
                      f"Time={elapsed:.1f}m | ETA={eta:.1f}m")

            # ========== Save Checkpoints ==========
            if iter_count % 10000 == 0:
                model_path = f'/content/drive/MyDrive/RFB-ESRGAN-Output/generator_iter_{iter_count}.pth'
                torch.save(generator.state_dict(), model_path)
                saved_models.append(model_path)
                print(f"✓ Checkpoint saved: {model_path}")
                torch.cuda.empty_cache()

    total_time = (time.time() - start_time) / 3600
    print(f"\n{'='*60}")
    print(f"✓ Training complete!")
    print(f"  Total time: {total_time:.2f} hours")
    print(f"  Checkpoints saved: {len(saved_models)}")
    print(f"{'='*60}")

    return saved_models


print("✓ Training function defined!")

✓ Training function defined!


## Step 7: Resume Training from Iteration 160,000

**Run this cell to start training!** It will continue from iteration 160,000 to 200,000.

In [8]:
# Start training from iteration 160,000
print("\n" + "="*60)
print("STARTING TRAINING RESUME")
print("="*60)

saved_models = train_stage2_resume(
    generator=generator,
    discriminator=discriminator,
    train_loader=train_loader,
    val_loader=val_loader,
    total_iterations=wandb.config.stage2_iters,  # 200,000
    start_iter=wandb.config.start_iter,          # 160,000
    lr=wandb.config.stage2_lr
)

print(f"\n✓ Training resumed and completed!")
print(f"  Saved {len(saved_models)} new checkpoints:")
for model_path in saved_models:
    print(f"    - {model_path}")


STARTING TRAINING RESUME

STAGE 2: RESUMING GAN TRAINING
Starting from iteration: 160000
Target iteration: 200000
Remaining iterations: 40000


NameError: name 'VGGPerceptualLoss' is not defined

## ⚠️ STOP! Dataset Issue Detected

**DO NOT CONTINUE TRAINING YET!** The warnings above mean patches aren't being found. Let's diagnose the problem first.

In [ ]:
# DIAGNOSTIC: Check how many patches are actually failing
print("🔍 Diagnosing Dataset Loading Issues...")
print("=" * 70)

# Check failed count from dataset
if hasattr(train_dataset, 'failed_count'):
    total_patches = len(train_dataset)
    failed_patches = train_dataset.failed_count
    success_rate = ((total_patches - failed_patches) / total_patches) * 100
    
    print(f"Total patches in dataset: {total_patches}")
    print(f"Failed to load: {failed_patches}")
    print(f"Success rate: {success_rate:.2f}%")
    
    if success_rate < 50:
        print("\n❌ CRITICAL: More than 50% of patches are failing!")
        print("   This will produce a useless model trained on random noise.")
        print("\n🔧 RECOMMENDED ACTIONS:")
        print("   1. Check if BigEarthNet-S2 dataset is fully downloaded")
        print("   2. Verify the CSV patch names match the actual directory structure")
        print("   3. Inspect a few patch directories to understand the naming")
        raise ValueError("Dataset loading failure rate too high - cannot continue training")
    elif success_rate < 90:
        print(f"\n⚠️  WARNING: {100-success_rate:.1f}% patch failure rate is high")
        print("   Training quality will be degraded.")
else:
    print("⚠️  Cannot determine failure rate (old dataset implementation)")

# Let's inspect the actual dataset structure
print("\n" + "=" * 70)
print("📁 DATASET STRUCTURE INSPECTION")
print("=" * 70)

import os
import glob

# Check what's actually in the BigEarthNet directory
if os.path.exists(DATASET_ROOT):
    # Get a sample of actual patch directories
    sample_patches = []
    for root, dirs, files in os.walk(DATASET_ROOT):
        if any(f.endswith('.tif') for f in files):
            sample_patches.append(root)
        if len(sample_patches) >= 5:
            break
    
    print(f"\nActual patches found in {DATASET_ROOT}:")
    for i, patch in enumerate(sample_patches[:5], 1):
        patch_name = os.path.basename(patch)
        print(f"  {i}. {patch_name}")
        # List some .tif files in this patch
        tif_files = glob.glob(os.path.join(patch, "*.tif"))[:3]
        for tif in tif_files:
            print(f"      - {os.path.basename(tif)}")
    
    # Compare with what CSV expects
    print(f"\nCSV expects patches like:")
    if train_df is not None:
        for i in range(min(5, len(train_df))):
            expected_name = str(train_df.iloc[i][train_dataset.path_column])
            print(f"  {i+1}. {expected_name}")
    
    print("\n💡 TIP: The patch names in CSV must match the actual directory names!")
    print("   If they don't match, we need to fix the path matching logic.")
else:
    print(f"❌ ERROR: Dataset root not found at {DATASET_ROOT}")
    raise FileNotFoundError(f"BigEarthNet dataset not accessible")

## Step 8: Create Ensemble Model

Average the last 10 checkpoints to create a robust ensemble model.

In [ ]:
def create_ensemble(generator, model_paths, top_k=10):
    """
    Create ensemble by averaging parameters of top-k checkpoints.
    """
    print(f"\n{'='*60}")
    print(f"Creating ensemble from top-{top_k} models...")
    print(f"{'='*60}")

    # Select top-k models (use last k models)
    selected_models = model_paths[-top_k:] if len(model_paths) >= top_k else model_paths

    print(f"\nAveraging {len(selected_models)} models:")
    for i, path in enumerate(selected_models, 1):
        iter_num = path.split('_')[-1].replace('.pth', '')
        print(f"  {i}. Iteration {iter_num}")

    # Average state dicts
    ensemble_state = OrderedDict()
    for path in tqdm(selected_models, desc="Averaging models"):
        state = torch.load(path, map_location=device)
        for key in state:
            if key not in ensemble_state:
                ensemble_state[key] = state[key].clone()
            else:
                ensemble_state[key] += state[key]

    # Divide by number of models
    for key in ensemble_state:
        ensemble_state[key] /= len(selected_models)

    # Load into generator
    generator.load_state_dict(ensemble_state)

    # Save ensemble model
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_ensemble_final.pth'
    torch.save(ensemble_state, save_path)

    print(f"\n✓ Ensemble model created and saved!")
    print(f"  Path: {save_path}")
    print(f"  Models averaged: {len(selected_models)}")
    print(f"{'='*60}")

    return save_path


# Create ensemble from all available checkpoints
# Include both new checkpoints and existing ones if you want
all_checkpoints = [
    '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_iter_160000.pth',
] + saved_models

ensemble_path = create_ensemble(
    generator=generator,
    model_paths=all_checkpoints,
    top_k=wandb.config.ensemble_models
)

print(f"\n🎉 Final ensemble model ready at: {ensemble_path}")

## Step 9: Test the Resumed Model (Optional)

Quick test to visualize results from the resumed training.

In [ ]:
import matplotlib.pyplot as plt

def test_model(generator, val_loader, device, num_samples=3):
    """Quick visual test of the model"""
    generator.eval()

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 4))

    with torch.no_grad():
        for i, (lr_img, hr_img) in enumerate(val_loader):
            if i >= num_samples:
                break

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)
            sr_img = generator(lr_img)

            # Convert to numpy for visualization
            lr_np = ((lr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_np = ((sr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            hr_np = ((hr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()

            axes[i, 0].imshow(lr_np)
            axes[i, 0].set_title(f'LR Input ({lr_img.shape[2]}x{lr_img.shape[3]})')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(sr_np)
            axes[i, 1].set_title('Super-Resolved (Ours)')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(hr_np)
            axes[i, 2].set_title(f'HR Ground Truth ({hr_img.shape[2]}x{hr_img.shape[3]})')
            axes[i, 2].axis('off')

    plt.tight_layout()
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/test_results_resumed.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    wandb.log({"test_results": wandb.Image(plt)})
    plt.show()

    print(f"\n✓ Test results saved to: {save_path}")
    generator.train()


# Test the model
test_model(generator, val_loader, device, num_samples=3)

## Step 10: Finish WandB Run

In [ ]:
# Finish WandB logging
wandb.finish()

print("\n" + "="*60)
print("✅ TRAINING RESUME COMPLETE!")
print("="*60)
print("\n📊 Summary:")
print(f"  ✓ Resumed from iteration: 160,000")
print(f"  ✓ Completed to iteration: 200,000")
print(f"  ✓ New checkpoints saved: {len(saved_models)}")
print(f"  ✓ Final ensemble model: {ensemble_path}")
print("\n📁 All files saved to: /content/drive/MyDrive/RFB-ESRGAN-Output/")
print(f"\n🌐 WandB Dashboard: Check your WandB project for training metrics")
print("\n" + "="*60)

## Step 11: Comprehensive Model Evaluation

Now let's thoroughly evaluate the model with advanced metrics and comparisons against baselines.

In [ ]:
# ========== COMPREHENSIVE EVALUATION METRICS ==========

# Install additional packages if needed
!pip install -q lpips pytorch-msssim scikit-learn seaborn

import lpips
from pytorch_msssim import ssim, ms_ssim
from sklearn.metrics import confusion_matrix, cohen_kappa_score, precision_recall_fscore_support, top_k_accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict

print("✓ Evaluation packages installed!")


# ========== 1. SUPER-RESOLUTION METRICS ==========

class SuperResolutionMetrics:
    """Comprehensive SR evaluation metrics"""
    def __init__(self, device):
        self.device = device
        # LPIPS loss network (Alex)
        self.lpips_fn = lpips.LPIPS(net='alex').to(device)

    def calculate_psnr(self, sr, hr):
        """Peak Signal-to-Noise Ratio"""
        mse = F.mse_loss(sr, hr)
        psnr = 10 * torch.log10(4 / mse)  # Range [-1,1] → max=2, so 4
        return psnr.item()

    def calculate_ssim(self, sr, hr):
        """Structural Similarity Index"""
        # Normalize from [-1,1] to [0,1]
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ssim_val = ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ssim_val.item()

    def calculate_ms_ssim(self, sr, hr):
        """Multi-Scale Structural Similarity Index"""
        sr_norm = (sr + 1) / 2
        hr_norm = (hr + 1) / 2
        ms_ssim_val = ms_ssim(sr_norm, hr_norm, data_range=1.0, size_average=True)
        return ms_ssim_val.item()

    def calculate_lpips(self, sr, hr):
        """Learned Perceptual Image Patch Similarity"""
        lpips_val = self.lpips_fn(sr, hr)
        return lpips_val.mean().item()

    def calculate_mae(self, sr, hr):
        """Mean Absolute Error"""
        mae = F.l1_loss(sr, hr)
        return mae.item()

    def calculate_rmse(self, sr, hr):
        """Root Mean Square Error"""
        mse = F.mse_loss(sr, hr)
        rmse = torch.sqrt(mse)
        return rmse.item()

    def calculate_ndvi_error(self, sr, hr):
        """Spectral Consistency - NDVI Error for vegetation index accuracy"""
        # Extract red channel (assuming channel 0 is red after normalization)
        sr_red = sr[:, 0:1, :, :]  # Red channel
        hr_red = hr[:, 0:1, :, :]

        # Simplified NDVI approximation
        ndvi_error = F.l1_loss(sr_red, hr_red)
        return ndvi_error.item()

    def calculate_edge_preservation(self, sr, hr):
        """Edge Preservation using Sobel filters"""
        # Simple edge detection using convolution
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(self.device)
        sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3).to(self.device)

        # Average across channels
        sr_gray = sr.mean(dim=1, keepdim=True)
        hr_gray = hr.mean(dim=1, keepdim=True)

        # Apply Sobel filters
        sr_edge_x = F.conv2d(sr_gray, sobel_x, padding=1)
        sr_edge_y = F.conv2d(sr_gray, sobel_y, padding=1)
        hr_edge_x = F.conv2d(hr_gray, sobel_x, padding=1)
        hr_edge_y = F.conv2d(hr_gray, sobel_y, padding=1)

        sr_edge = torch.sqrt(sr_edge_x**2 + sr_edge_y**2)
        hr_edge = torch.sqrt(hr_edge_x**2 + hr_edge_y**2)

        edge_error = F.l1_loss(sr_edge, hr_edge)
        return edge_error.item()

    def evaluate_batch(self, sr, hr):
        """Evaluate all SR metrics on a batch"""
        metrics = {
            'psnr': self.calculate_psnr(sr, hr),
            'ssim': self.calculate_ssim(sr, hr),
            'ms_ssim': self.calculate_ms_ssim(sr, hr),
            'lpips': self.calculate_lpips(sr, hr),
            'mae': self.calculate_mae(sr, hr),
            'rmse': self.calculate_rmse(sr, hr),
            'ndvi_error': self.calculate_ndvi_error(sr, hr),
            'edge_preservation': self.calculate_edge_preservation(sr, hr)
        }
        return metrics


# ========== 2. BASELINE COMPARISON MODELS ==========

class BicubicUpsampler:
    """Baseline bicubic interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)


class BilinearUpsampler:
    """Baseline bilinear interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='bilinear', align_corners=False)


class NearestUpsampler:
    """Baseline nearest neighbor interpolation"""
    def __init__(self, scale_factor=8):
        self.scale_factor = scale_factor

    def __call__(self, lr_img):
        return F.interpolate(lr_img, scale_factor=self.scale_factor, mode='nearest')


class SimpleSRCNN(nn.Module):
    """Lightweight SRCNN baseline for comparison"""
    def __init__(self, scale_factor=8):
        super(SimpleSRCNN, self).__init__()
        self.scale_factor = scale_factor

        # SRCNN: 3 conv layers
        self.conv1 = nn.Conv2d(3, 64, 9, padding=4)
        self.conv2 = nn.Conv2d(64, 32, 1, padding=0)
        self.conv3 = nn.Conv2d(32, 3, 5, padding=2)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # Bicubic upsampling first
        x = F.interpolate(x, scale_factor=self.scale_factor, mode='bicubic', align_corners=False)
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.conv3(x)
        return torch.tanh(x)


print("✓ Evaluation metrics and baseline models defined!")

In [ ]:
# ========== 3. COMPREHENSIVE COMPARATIVE EVALUATION ==========

def comparative_evaluation(generator, val_loader, device, num_samples=100):
    """Compare RFB-ESRGAN against multiple baselines with comprehensive metrics"""
    print("\n" + "="*70)
    print("COMPARATIVE EVALUATION: RFB-ESRGAN vs. Baselines")
    print("="*70)

    # Initialize models and metrics
    sr_metrics = SuperResolutionMetrics(device)
    bicubic = BicubicUpsampler(scale_factor=8)
    bilinear = BilinearUpsampler(scale_factor=8)
    nearest = NearestUpsampler(scale_factor=8)
    srcnn = SimpleSRCNN(scale_factor=8).to(device)
    srcnn.eval()

    # Results storage
    model_names = ['nearest', 'bilinear', 'bicubic', 'srcnn', 'ours']
    results = {name: defaultdict(list) for name in model_names}
    inference_times = {name: [] for name in model_names}

    generator.eval()
    sample_count = 0

    print(f"\n📊 Evaluating on {num_samples} samples...")
    print(f"Models: Nearest, Bilinear, Bicubic, SRCNN, RFB-ESRGAN (Ours)")

    with torch.no_grad():
        for lr_img, hr_img in tqdm(val_loader, desc="Evaluating"):
            if sample_count >= num_samples:
                break

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            # ========== Nearest Neighbor ==========
            start_time = time.time()
            sr_nearest = nearest(lr_img)
            inference_times['nearest'].append(time.time() - start_time)
            metrics_nearest = sr_metrics.evaluate_batch(sr_nearest, hr_img)
            for k, v in metrics_nearest.items():
                results['nearest'][k].append(v)

            # ========== Bilinear ==========
            start_time = time.time()
            sr_bilinear = bilinear(lr_img)
            inference_times['bilinear'].append(time.time() - start_time)
            metrics_bilinear = sr_metrics.evaluate_batch(sr_bilinear, hr_img)
            for k, v in metrics_bilinear.items():
                results['bilinear'][k].append(v)

            # ========== Bicubic ==========
            start_time = time.time()
            sr_bicubic = bicubic(lr_img)
            inference_times['bicubic'].append(time.time() - start_time)
            metrics_bicubic = sr_metrics.evaluate_batch(sr_bicubic, hr_img)
            for k, v in metrics_bicubic.items():
                results['bicubic'][k].append(v)

            # ========== SRCNN ==========
            start_time = time.time()
            sr_srcnn = srcnn(lr_img)
            inference_times['srcnn'].append(time.time() - start_time)
            metrics_srcnn = sr_metrics.evaluate_batch(sr_srcnn, hr_img)
            for k, v in metrics_srcnn.items():
                results['srcnn'][k].append(v)

            # ========== RFB-ESRGAN (Ours) ==========
            start_time = time.time()
            sr_ours = generator(lr_img)
            inference_times['ours'].append(time.time() - start_time)
            metrics_ours = sr_metrics.evaluate_batch(sr_ours, hr_img)
            for k, v in metrics_ours.items():
                results['ours'][k].append(v)

            sample_count += lr_img.size(0)

    # ========== Calculate Average Metrics ==========
    print("\n" + "="*70)
    print("RESULTS SUMMARY")
    print("="*70)

    comparison_table = []

    for model_name in model_names:
        avg_metrics = {k: np.mean(v) for k, v in results[model_name].items()}
        avg_time = np.mean(inference_times[model_name]) * 1000  # ms

        print(f"\n{model_name.upper()}:")
        print(f"  PSNR: {avg_metrics['psnr']:.2f} dB")
        print(f"  SSIM: {avg_metrics['ssim']:.4f}")
        print(f"  MS-SSIM: {avg_metrics['ms_ssim']:.4f}")
        print(f"  LPIPS: {avg_metrics['lpips']:.4f} (lower is better)")
        print(f"  MAE: {avg_metrics['mae']:.4f}")
        print(f"  RMSE: {avg_metrics['rmse']:.4f}")
        print(f"  NDVI Error: {avg_metrics['ndvi_error']:.4f}")
        print(f"  Edge Preservation: {avg_metrics['edge_preservation']:.4f}")
        print(f"  Inference Time: {avg_time:.2f} ms/image")

        comparison_table.append({
            'model': model_name,
            **avg_metrics,
            'inference_time_ms': avg_time
        })

    # ========== Calculate Improvement Deltas ==========
    print("\n" + "="*70)
    print("IMPROVEMENT vs. BASELINES")
    print("="*70)

    ours_psnr = np.mean(results['ours']['psnr'])
    ours_ssim = np.mean(results['ours']['ssim'])
    bicubic_psnr = np.mean(results['bicubic']['psnr'])
    bicubic_ssim = np.mean(results['bicubic']['ssim'])
    srcnn_psnr = np.mean(results['srcnn']['psnr'])
    srcnn_ssim = np.mean(results['srcnn']['ssim'])

    delta_psnr_bicubic = ours_psnr - bicubic_psnr
    delta_ssim_bicubic = ours_ssim - bicubic_ssim
    delta_psnr_srcnn = ours_psnr - srcnn_psnr
    delta_ssim_srcnn = ours_ssim - srcnn_ssim

    print(f"\nΔPSNR vs. Bicubic: +{delta_psnr_bicubic:.2f} dB ({delta_psnr_bicubic/bicubic_psnr*100:.1f}% improvement)")
    print(f"ΔSSIM vs. Bicubic: +{delta_ssim_bicubic:.4f} ({delta_ssim_bicubic/bicubic_ssim*100:.1f}% improvement)")
    print(f"ΔPSNR vs. SRCNN: +{delta_psnr_srcnn:.2f} dB ({delta_psnr_srcnn/srcnn_psnr*100:.1f}% improvement)")
    print(f"ΔSSIM vs. SRCNN: +{delta_ssim_srcnn:.4f} ({delta_ssim_srcnn/srcnn_ssim*100:.1f}% improvement)")

    # ========== Model Efficiency ==========
    print("\n" + "="*70)
    print("MODEL EFFICIENCY METRICS")
    print("="*70)

    # Parameter count
    def count_parameters(model):
        if isinstance(model, nn.DataParallel):
            return sum(p.numel() for p in model.module.parameters())
        return sum(p.numel() for p in model.parameters())

    ours_params = count_parameters(generator)
    srcnn_params = count_parameters(srcnn)

    print(f"\nParameter Count:")
    print(f"  RFB-ESRGAN (Ours): {ours_params/1e6:.2f}M parameters")
    print(f"  SRCNN: {srcnn_params/1e6:.2f}M parameters")
    print(f"  Bicubic/Bilinear/Nearest: 0M parameters (no learning)")

    # Parameter efficiency
    psnr_per_param_ours = (ours_psnr - bicubic_psnr) / (ours_params / 1e6)
    psnr_per_param_srcnn = (srcnn_psnr - bicubic_psnr) / (srcnn_params / 1e6)

    print(f"\nParameter Efficiency (ΔPSNR per 1M params vs. Bicubic):")
    print(f"  RFB-ESRGAN: {psnr_per_param_ours:.3f} dB/M")
    print(f"  SRCNN: {psnr_per_param_srcnn:.3f} dB/M")

    # ========== System Performance Metrics ==========
    print("\n" + "="*70)
    print("SYSTEM PERFORMANCE METRICS")
    print("="*70)

    avg_time_ours = np.mean(inference_times['ours'])
    fps_ours = 1.0 / avg_time_ours if avg_time_ours > 0 else 0

    print(f"\nInference Performance (Ours):")
    print(f"  Latency: {avg_time_ours*1000:.2f} ms/image")
    print(f"  Throughput: {fps_ours:.2f} FPS")
    print(f"  Speed vs. Bicubic: {np.mean(inference_times['bicubic'])/avg_time_ours:.2f}x slower")

    # GPU Memory footprint
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        with torch.no_grad():
            _ = generator(lr_img)
        memory_allocated = torch.cuda.max_memory_allocated() / 1024**2  # MB
        print(f"  GPU Memory Footprint: {memory_allocated:.2f} MB")

    # ========== Statistical Significance ==========
    print("\n" + "="*70)
    print("STATISTICAL ANALYSIS")
    print("="*70)

    from scipy import stats

    # T-test comparing our model vs bicubic
    t_stat, p_value = stats.ttest_rel(results['ours']['psnr'], results['bicubic']['psnr'])
    print(f"\nPSNR T-test (Ours vs. Bicubic):")
    print(f"  t-statistic: {t_stat:.3f}")
    print(f"  p-value: {p_value:.6f}")
    print(f"  Statistically significant: {'Yes' if p_value < 0.05 else 'No'} (p < 0.05)")

    # Standard deviations
    print(f"\nStandard Deviations:")
    for model_name in ['bicubic', 'srcnn', 'ours']:
        std_psnr = np.std(results[model_name]['psnr'])
        std_ssim = np.std(results[model_name]['ssim'])
        print(f"  {model_name.upper()}: PSNR±{std_psnr:.2f}, SSIM±{std_ssim:.4f}")

    # ========== Log to WandB ==========
    wandb.log({
        'eval/psnr_ours': ours_psnr,
        'eval/ssim_ours': ours_ssim,
        'eval/ms_ssim_ours': np.mean(results['ours']['ms_ssim']),
        'eval/lpips_ours': np.mean(results['ours']['lpips']),
        'eval/mae_ours': np.mean(results['ours']['mae']),
        'eval/rmse_ours': np.mean(results['ours']['rmse']),
        'eval/psnr_bicubic': bicubic_psnr,
        'eval/ssim_bicubic': bicubic_ssim,
        'eval/psnr_srcnn': srcnn_psnr,
        'eval/ssim_srcnn': srcnn_ssim,
        'eval/delta_psnr_vs_bicubic': delta_psnr_bicubic,
        'eval/delta_ssim_vs_bicubic': delta_ssim_bicubic,
        'eval/delta_psnr_vs_srcnn': delta_psnr_srcnn,
        'eval/delta_ssim_vs_srcnn': delta_ssim_srcnn,
        'eval/inference_time_ms': avg_time_ours * 1000,
        'eval/throughput_fps': fps_ours,
        'eval/parameters_millions': ours_params / 1e6,
        'eval/psnr_per_param': psnr_per_param_ours,
        'eval/t_test_p_value': p_value,
    })

    return comparison_table, results, inference_times


print("✓ Comparative evaluation function defined!")

In [ ]:
# ========== 4. COMPREHENSIVE VISUALIZATIONS ==========

def create_comparison_visualizations(results, inference_times, comparison_table):
    """Create comprehensive comparison visualizations"""
    print("\n📊 Creating comprehensive visualizations...")

    model_names = ['nearest', 'bilinear', 'bicubic', 'srcnn', 'ours']
    display_names = ['Nearest', 'Bilinear', 'Bicubic', 'SRCNN', 'RFB-ESRGAN']
    colors = ['#d62728', '#ff7f0e', '#2ca02c', '#9467bd', '#1f77b4']

    fig = plt.figure(figsize=(20, 12))

    # Plot 1: PSNR Comparison
    ax1 = plt.subplot(3, 4, 1)
    psnr_values = [np.mean(results[m]['psnr']) for m in model_names]
    bars = ax1.bar(display_names, psnr_values, color=colors)
    ax1.set_ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
    ax1.set_title('Peak Signal-to-Noise Ratio', fontsize=14, fontweight='bold')
    ax1.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 2: SSIM Comparison
    ax2 = plt.subplot(3, 4, 2)
    ssim_values = [np.mean(results[m]['ssim']) for m in model_names]
    bars = ax2.bar(display_names, ssim_values, color=colors)
    ax2.set_ylabel('SSIM', fontsize=12, fontweight='bold')
    ax2.set_title('Structural Similarity Index', fontsize=14, fontweight='bold')
    ax2.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 3: MS-SSIM Comparison
    ax3 = plt.subplot(3, 4, 3)
    ms_ssim_values = [np.mean(results[m]['ms_ssim']) for m in model_names]
    bars = ax3.bar(display_names, ms_ssim_values, color=colors)
    ax3.set_ylabel('MS-SSIM', fontsize=12, fontweight='bold')
    ax3.set_title('Multi-Scale SSIM', fontsize=14, fontweight='bold')
    ax3.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax3.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 4: LPIPS Comparison (lower is better)
    ax4 = plt.subplot(3, 4, 4)
    lpips_values = [np.mean(results[m]['lpips']) for m in model_names]
    bars = ax4.bar(display_names, lpips_values, color=colors)
    ax4.set_ylabel('LPIPS (lower=better)', fontsize=12, fontweight='bold')
    ax4.set_title('Learned Perceptual Quality', fontsize=14, fontweight='bold')
    ax4.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax4.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 5: MAE Comparison (lower is better)
    ax5 = plt.subplot(3, 4, 5)
    mae_values = [np.mean(results[m]['mae']) for m in model_names]
    bars = ax5.bar(display_names, mae_values, color=colors)
    ax5.set_ylabel('MAE (lower=better)', fontsize=12, fontweight='bold')
    ax5.set_title('Mean Absolute Error', fontsize=14, fontweight='bold')
    ax5.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax5.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 6: RMSE Comparison (lower is better)
    ax6 = plt.subplot(3, 4, 6)
    rmse_values = [np.mean(results[m]['rmse']) for m in model_names]
    bars = ax6.bar(display_names, rmse_values, color=colors)
    ax6.set_ylabel('RMSE (lower=better)', fontsize=12, fontweight='bold')
    ax6.set_title('Root Mean Square Error', fontsize=14, fontweight='bold')
    ax6.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax6.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 7: Inference Time
    ax7 = plt.subplot(3, 4, 7)
    times = [np.mean(inference_times[m])*1000 for m in model_names]
    bars = ax7.bar(display_names, times, color=colors)
    ax7.set_ylabel('Time (ms)', fontsize=12, fontweight='bold')
    ax7.set_title('Inference Latency', fontsize=14, fontweight='bold')
    ax7.grid(axis='y', alpha=0.3)
    ax7.set_yscale('log')
    for bar in bars:
        height = bar.get_height()
        ax7.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 8: Edge Preservation
    ax8 = plt.subplot(3, 4, 8)
    edge_values = [np.mean(results[m]['edge_preservation']) for m in model_names]
    bars = ax8.bar(display_names, edge_values, color=colors)
    ax8.set_ylabel('Edge Error (lower=better)', fontsize=12, fontweight='bold')
    ax8.set_title('Edge Preservation', fontsize=14, fontweight='bold')
    ax8.grid(axis='y', alpha=0.3)
    for bar in bars:
        height = bar.get_height()
        ax8.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.4f}', ha='center', va='bottom', fontsize=9)
    plt.xticks(rotation=45, ha='right')

    # Plot 9: PSNR Box Plot
    ax9 = plt.subplot(3, 4, 9)
    psnr_data = [results[m]['psnr'] for m in model_names]
    bp = ax9.boxplot(psnr_data, labels=display_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax9.set_ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
    ax9.set_title('PSNR Distribution', fontsize=14, fontweight='bold')
    ax9.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')

    # Plot 10: SSIM Box Plot
    ax10 = plt.subplot(3, 4, 10)
    ssim_data = [results[m]['ssim'] for m in model_names]
    bp = ax10.boxplot(ssim_data, labels=display_names, patch_artist=True)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
    ax10.set_ylabel('SSIM', fontsize=12, fontweight='bold')
    ax10.set_title('SSIM Distribution', fontsize=14, fontweight='bold')
    ax10.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=45, ha='right')

    # Plot 11: Overall Performance Radar Chart
    ax11 = plt.subplot(3, 4, 11, projection='polar')
    categories = ['PSNR', 'SSIM', 'MS-SSIM', 'Speed\n(inv)', 'LPIPS\n(inv)']

    def normalize(values):
        min_val, max_val = min(values), max(values)
        return [(v - min_val) / (max_val - min_val) if max_val > min_val else 0.5 for v in values]

    # Only plot our model vs best baseline (bicubic)
    for idx, model_name in enumerate(['bicubic', 'ours']):
        radar_values = [
            normalize(psnr_values)[model_names.index(model_name)],
            normalize(ssim_values)[model_names.index(model_name)],
            normalize(ms_ssim_values)[model_names.index(model_name)],
            1 - normalize(times)[model_names.index(model_name)],  # Invert (faster is better)
            1 - normalize(lpips_values)[model_names.index(model_name)]  # Invert (lower is better)
        ]

        angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
        radar_values += radar_values[:1]
        angles += angles[:1]

        color = colors[2] if model_name == 'bicubic' else colors[4]
        label = 'Bicubic' if model_name == 'bicubic' else 'RFB-ESRGAN'
        ax11.plot(angles, radar_values, 'o-', linewidth=2, label=label, color=color)
        ax11.fill(angles, radar_values, alpha=0.15, color=color)

    ax11.set_xticks(angles[:-1])
    ax11.set_xticklabels(categories, fontsize=10)
    ax11.set_ylim(0, 1)
    ax11.set_title('Overall Performance\n(Normalized)', fontsize=14, fontweight='bold', pad=20)
    ax11.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    ax11.grid(True)

    # Plot 12: Quality vs Speed Trade-off
    ax12 = plt.subplot(3, 4, 12)
    for i, model_name in enumerate(model_names):
        ax12.scatter(times[i], psnr_values[i], s=200, c=colors[i],
                    label=display_names[i], alpha=0.7, edgecolors='black', linewidths=2)
        ax12.annotate(display_names[i], (times[i], psnr_values[i]),
                     xytext=(5, 5), textcoords='offset points', fontsize=9)
    ax12.set_xlabel('Inference Time (ms)', fontsize=12, fontweight='bold')
    ax12.set_ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
    ax12.set_title('Quality vs Speed Trade-off', fontsize=14, fontweight='bold')
    ax12.set_xscale('log')
    ax12.grid(True, alpha=0.3)
    ax12.legend(loc='best', fontsize=8)

    plt.tight_layout()
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/comprehensive_evaluation.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    wandb.log({"comprehensive_evaluation": wandb.Image(plt)})
    plt.show()

    print(f"✓ Comprehensive visualizations saved to: {save_path}")


print("✓ Visualization functions defined!")

In [ ]:
# ========== 5. VISUAL QUALITY COMPARISON ==========

def visualize_quality_comparison(generator, val_loader, device, num_samples=5):
    """Visualize SR results for visual quality assessment across all methods"""
    print("\n📸 Generating visual quality samples...")

    bicubic = BicubicUpsampler(scale_factor=8)
    srcnn = SimpleSRCNN(scale_factor=8).to(device)
    srcnn.eval()
    generator.eval()

    fig, axes = plt.subplots(num_samples, 5, figsize=(20, num_samples * 4))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for i, (lr_img, hr_img) in enumerate(val_loader):
            if i >= num_samples:
                break

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            # Generate SR images
            sr_bicubic = bicubic(lr_img)
            sr_srcnn = srcnn(lr_img)
            sr_ours = generator(lr_img)

            # Take first image in batch
            lr_np = ((lr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            hr_np = ((hr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_bicubic_np = ((sr_bicubic[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_srcnn_np = ((sr_srcnn[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_ours_np = ((sr_ours[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()

            # Calculate metrics for annotations
            psnr_bicubic = 10 * torch.log10(4 / F.mse_loss(sr_bicubic[0:1], hr_img[0:1])).item()
            psnr_srcnn = 10 * torch.log10(4 / F.mse_loss(sr_srcnn[0:1], hr_img[0:1])).item()
            psnr_ours = 10 * torch.log10(4 / F.mse_loss(sr_ours[0:1], hr_img[0:1])).item()

            # Plot
            axes[i, 0].imshow(lr_np)
            axes[i, 0].set_title(f'LR Input\n{lr_img.shape[2]}x{lr_img.shape[3]}', fontsize=12, fontweight='bold')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(sr_bicubic_np)
            axes[i, 1].set_title(f'Bicubic\nPSNR: {psnr_bicubic:.2f} dB', fontsize=12, fontweight='bold')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(sr_srcnn_np)
            axes[i, 2].set_title(f'SRCNN\nPSNR: {psnr_srcnn:.2f} dB', fontsize=12, fontweight='bold')
            axes[i, 2].axis('off')

            axes[i, 3].imshow(sr_ours_np)
            axes[i, 3].set_title(f'RFB-ESRGAN (Ours)\nPSNR: {psnr_ours:.2f} dB', fontsize=12, fontweight='bold', color='blue')
            axes[i, 3].axis('off')

            axes[i, 4].imshow(hr_np)
            axes[i, 4].set_title(f'HR Ground Truth\n{hr_img.shape[2]}x{hr_img.shape[3]}', fontsize=12, fontweight='bold')
            axes[i, 4].axis('off')

    plt.tight_layout()
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/visual_quality_samples.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    wandb.log({"visual_quality_samples": wandb.Image(plt)})
    plt.show()

    print(f"✓ Visual samples saved to: {save_path}")
    generator.train()


# ========== 6. DETAILED DIFFERENCE MAPS ==========

def visualize_difference_maps(generator, val_loader, device, num_samples=3):
    """Visualize pixel-wise difference maps to show improvements"""
    print("\n🔍 Generating difference maps...")

    bicubic = BicubicUpsampler(scale_factor=8)
    generator.eval()

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    if num_samples == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for i, (lr_img, hr_img) in enumerate(val_loader):
            if i >= num_samples:
                break

            lr_img = lr_img.to(device)
            hr_img = hr_img.to(device)

            sr_bicubic = bicubic(lr_img)
            sr_ours = generator(lr_img)

            # Calculate difference maps
            diff_bicubic = torch.abs(sr_bicubic[0] - hr_img[0])
            diff_ours = torch.abs(sr_ours[0] - hr_img[0])

            # Convert to numpy
            hr_np = ((hr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            sr_ours_np = ((sr_ours[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
            diff_bicubic_np = diff_bicubic.cpu().permute(1, 2, 0).numpy().mean(axis=2)
            diff_ours_np = diff_ours.cpu().permute(1, 2, 0).numpy().mean(axis=2)

            # Plot
            axes[i, 0].imshow(hr_np)
            axes[i, 0].set_title('HR Ground Truth', fontsize=12, fontweight='bold')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(sr_ours_np)
            axes[i, 1].set_title('RFB-ESRGAN (Ours)', fontsize=12, fontweight='bold')
            axes[i, 1].axis('off')

            im2 = axes[i, 2].imshow(diff_bicubic_np, cmap='hot', vmin=0, vmax=0.5)
            axes[i, 2].set_title(f'Bicubic Error Map\nMAE: {diff_bicubic_np.mean():.4f}', fontsize=12, fontweight='bold')
            axes[i, 2].axis('off')
            plt.colorbar(im2, ax=axes[i, 2], fraction=0.046)

            im3 = axes[i, 3].imshow(diff_ours_np, cmap='hot', vmin=0, vmax=0.5)
            axes[i, 3].set_title(f'Our Error Map\nMAE: {diff_ours_np.mean():.4f}', fontsize=12, fontweight='bold', color='blue')
            axes[i, 3].axis('off')
            plt.colorbar(im3, ax=axes[i, 3], fraction=0.046)

    plt.tight_layout()
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/difference_maps.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    wandb.log({"difference_maps": wandb.Image(plt)})
    plt.show()

    print(f"✓ Difference maps saved to: {save_path}")
    generator.train()


print("✓ All visualization functions defined!")

## Step 12: Execute Comprehensive Evaluation

**Run this cell to perform complete model evaluation** with all metrics, visualizations, and comparisons.

In [ ]:
# ========== EXECUTE COMPREHENSIVE EVALUATION ==========

print("\n" + "="*70)
print("🚀 STARTING COMPREHENSIVE MODEL EVALUATION")
print("="*70)
print("\nThis will evaluate:")
print("  • 8 different metrics (PSNR, SSIM, MS-SSIM, LPIPS, MAE, RMSE, NDVI, Edge)")
print("  • 5 different models (Nearest, Bilinear, Bicubic, SRCNN, RFB-ESRGAN)")
print("  • Statistical significance tests")
print("  • 12 comprehensive visualizations")
print("  • Visual quality samples")
print("  • Difference/error maps")
print("="*70)

# Load the ensemble model if not already loaded
try:
    # Try to use the ensemble model from Step 9
    ensemble_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/generator_ensemble_final.pth'
    if os.path.exists(ensemble_path):
        print(f"\n📦 Loading ensemble model from: {ensemble_path}")
        ensemble_state = torch.load(ensemble_path, map_location=device)
        if isinstance(generator, nn.DataParallel):
            generator.module.load_state_dict(ensemble_state)
        else:
            generator.load_state_dict(ensemble_state)
        print("✓ Ensemble model loaded successfully!")
    else:
        print(f"\n⚠️  Ensemble not found, using current generator state")
except Exception as e:
    print(f"\n⚠️  Could not load ensemble, using current state: {e}")

# Run comprehensive evaluation (adjust num_samples based on your validation set size)
print("\n" + "="*70)
print("PHASE 1: QUANTITATIVE EVALUATION")
print("="*70)

comparison_table, results, inference_times = comparative_evaluation(
    generator=generator,
    val_loader=val_loader,
    device=device,
    num_samples=100  # Adjust this based on your validation set
)

# Create comprehensive visualizations
print("\n" + "="*70)
print("PHASE 2: CREATING VISUALIZATIONS")
print("="*70)

create_comparison_visualizations(results, inference_times, comparison_table)

# Generate visual quality samples
print("\n" + "="*70)
print("PHASE 3: VISUAL QUALITY ASSESSMENT")
print("="*70)

visualize_quality_comparison(
    generator=generator,
    val_loader=val_loader,
    device=device,
    num_samples=5
)

# Generate difference maps
print("\n" + "="*70)
print("PHASE 4: DIFFERENCE MAP ANALYSIS")
print("="*70)

visualize_difference_maps(
    generator=generator,
    val_loader=val_loader,
    device=device,
    num_samples=3
)

# ========== FINAL SUMMARY ==========
print("\n" + "="*70)
print("✅ COMPREHENSIVE EVALUATION COMPLETE!")
print("="*70)

print("\n📊 Evaluated Metrics:")
print("  ✓ PSNR (Peak Signal-to-Noise Ratio)")
print("  ✓ SSIM (Structural Similarity Index)")
print("  ✓ MS-SSIM (Multi-Scale SSIM)")
print("  ✓ LPIPS (Learned Perceptual Image Patch Similarity)")
print("  ✓ MAE (Mean Absolute Error)")
print("  ✓ RMSE (Root Mean Square Error)")
print("  ✓ NDVI Error (Spectral Consistency)")
print("  ✓ Edge Preservation Score")

print("\n🔬 Compared Against:")
print("  ✓ Nearest Neighbor Interpolation")
print("  ✓ Bilinear Interpolation")
print("  ✓ Bicubic Interpolation")
print("  ✓ SRCNN (Simple Super-Resolution CNN)")
print("  ✓ RFB-ESRGAN (Our Model)")

print("\n📈 Analysis Performed:")
print("  ✓ Statistical Significance (T-tests)")
print("  ✓ Performance Distribution (Box Plots)")
print("  ✓ Quality vs Speed Trade-off")
print("  ✓ Parameter Efficiency Analysis")
print("  ✓ System Performance Metrics")

print("\n📁 Results Saved to:")
print("  • /content/drive/MyDrive/RFB-ESRGAN-Output/comprehensive_evaluation.png")
print("  • /content/drive/MyDrive/RFB-ESRGAN-Output/visual_quality_samples.png")
print("  • /content/drive/MyDrive/RFB-ESRGAN-Output/difference_maps.png")

print(f"\n🌐 WandB Dashboard: {wandb.run.url}")
print("\n" + "="*70)

# Display summary table
print("\n📋 RESULTS SUMMARY TABLE:")
print("="*70)
import pandas as pd
df = pd.DataFrame(comparison_table)
df = df.round(4)
print(df.to_string(index=False))
print("="*70)

## Step 13: Additional Performance Analysis (Optional)

Extra analysis tools for deeper insights into model performance.

In [ ]:
# ========== ADDITIONAL PERFORMANCE ANALYSIS ==========

def analyze_model_convergence(results):
    """Analyze how metrics vary across the validation set"""
    print("\n" + "="*70)
    print("CONVERGENCE & STABILITY ANALYSIS")
    print("="*70)

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # PSNR over samples
    axes[0, 0].plot(results['ours']['psnr'], label='RFB-ESRGAN', linewidth=2, color='blue')
    axes[0, 0].plot(results['bicubic']['psnr'], label='Bicubic', linewidth=2, color='green', alpha=0.7)
    axes[0, 0].set_xlabel('Sample Index', fontsize=12, fontweight='bold')
    axes[0, 0].set_ylabel('PSNR (dB)', fontsize=12, fontweight='bold')
    axes[0, 0].set_title('PSNR Across Validation Samples', fontsize=14, fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # SSIM over samples
    axes[0, 1].plot(results['ours']['ssim'], label='RFB-ESRGAN', linewidth=2, color='blue')
    axes[0, 1].plot(results['bicubic']['ssim'], label='Bicubic', linewidth=2, color='green', alpha=0.7)
    axes[0, 1].set_xlabel('Sample Index', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('SSIM', fontsize=12, fontweight='bold')
    axes[0, 1].set_title('SSIM Across Validation Samples', fontsize=14, fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # PSNR Histogram
    axes[1, 0].hist(results['ours']['psnr'], bins=30, alpha=0.7, label='RFB-ESRGAN', color='blue')
    axes[1, 0].hist(results['bicubic']['psnr'], bins=30, alpha=0.7, label='Bicubic', color='green')
    axes[1, 0].set_xlabel('PSNR (dB)', fontsize=12, fontweight='bold')
    axes[1, 0].set_ylabel('Frequency', fontsize=12, fontweight='bold')
    axes[1, 0].set_title('PSNR Distribution', fontsize=14, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Improvement histogram
    improvements = np.array(results['ours']['psnr']) - np.array(results['bicubic']['psnr'])
    axes[1, 1].hist(improvements, bins=30, alpha=0.7, color='purple')
    axes[1, 1].axvline(improvements.mean(), color='red', linestyle='--', linewidth=2,
                       label=f'Mean: {improvements.mean():.2f} dB')
    axes[1, 1].set_xlabel('PSNR Improvement (dB)', fontsize=12, fontweight='bold')
    axes[1, 1].set_ylabel('Frequency', fontsize=12, fontweight='bold')
    axes[1, 1].set_title('PSNR Improvement Distribution', fontsize=14, fontweight='bold')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.tight_layout()
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/stability_analysis.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    wandb.log({"stability_analysis": wandb.Image(plt)})
    plt.show()

    print(f"\n✓ Stability analysis saved to: {save_path}")

    # Print statistics
    print(f"\nPSNR Improvement Statistics:")
    print(f"  Mean: {improvements.mean():.2f} dB")
    print(f"  Median: {np.median(improvements):.2f} dB")
    print(f"  Std Dev: {improvements.std():.2f} dB")
    print(f"  Min: {improvements.min():.2f} dB")
    print(f"  Max: {improvements.max():.2f} dB")
    print(f"  25th percentile: {np.percentile(improvements, 25):.2f} dB")
    print(f"  75th percentile: {np.percentile(improvements, 75):.2f} dB")

    # Consistency metric
    cv = improvements.std() / improvements.mean() * 100
    print(f"\n  Coefficient of Variation: {cv:.2f}%")
    print(f"  Consistency: {'High' if cv < 20 else 'Moderate' if cv < 40 else 'Low'}")


def analyze_failure_cases(generator, val_loader, device, results, top_k=3):
    """Identify and visualize worst-performing samples"""
    print("\n" + "="*70)
    print("FAILURE CASE ANALYSIS")
    print("="*70)

    # Find samples with lowest PSNR improvements
    improvements = np.array(results['ours']['psnr']) - np.array(results['bicubic']['psnr'])
    worst_indices = np.argsort(improvements)[:top_k]

    print(f"\nAnalyzing {top_k} samples with lowest PSNR improvement:")
    for idx in worst_indices:
        print(f"  Sample {idx}: Improvement = {improvements[idx]:.2f} dB")

    bicubic = BicubicUpsampler(scale_factor=8)
    generator.eval()

    fig, axes = plt.subplots(top_k, 4, figsize=(16, top_k * 4))
    if top_k == 1:
        axes = axes.reshape(1, -1)

    with torch.no_grad():
        for plot_idx, sample_idx in enumerate(worst_indices):
            # Get specific sample
            for i, (lr_img, hr_img) in enumerate(val_loader):
                if i == sample_idx:
                    lr_img = lr_img.to(device)
                    hr_img = hr_img.to(device)

                    sr_bicubic = bicubic(lr_img)
                    sr_ours = generator(lr_img)

                    # Convert to numpy
                    lr_np = ((lr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
                    hr_np = ((hr_img[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
                    sr_bicubic_np = ((sr_bicubic[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()
                    sr_ours_np = ((sr_ours[0].cpu() + 1) / 2).permute(1, 2, 0).numpy()

                    # Plot
                    axes[plot_idx, 0].imshow(lr_np)
                    axes[plot_idx, 0].set_title(f'LR Input (Sample {sample_idx})', fontsize=11, fontweight='bold')
                    axes[plot_idx, 0].axis('off')

                    axes[plot_idx, 1].imshow(sr_bicubic_np)
                    axes[plot_idx, 1].set_title(f'Bicubic\nPSNR: {results["bicubic"]["psnr"][sample_idx]:.2f} dB', fontsize=11, fontweight='bold')
                    axes[plot_idx, 1].axis('off')

                    axes[plot_idx, 2].imshow(sr_ours_np)
                    axes[plot_idx, 2].set_title(f'Ours\nPSNR: {results["ours"]["psnr"][sample_idx]:.2f} dB\nΔ: {improvements[sample_idx]:.2f} dB',
                                               fontsize=11, fontweight='bold', color='red')
                    axes[plot_idx, 2].axis('off')

                    axes[plot_idx, 3].imshow(hr_np)
                    axes[plot_idx, 3].set_title('HR Ground Truth', fontsize=11, fontweight='bold')
                    axes[plot_idx, 3].axis('off')
                    break

    plt.tight_layout()
    save_path = '/content/drive/MyDrive/RFB-ESRGAN-Output/failure_cases.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    wandb.log({"failure_cases": wandb.Image(plt)})
    plt.show()

    print(f"\n✓ Failure case analysis saved to: {save_path}")
    generator.train()


# Run additional analysis
print("\n🔍 Running additional performance analysis...")

analyze_model_convergence(results)
analyze_failure_cases(generator, val_loader, device, results, top_k=3)

print("\n✅ Additional analysis complete!")

## Step 14: Final WandB Cleanup

Close the WandB run and display final summary.

In [ ]:
# Finish WandB logging
wandb.finish()

print("\n" + "="*70)
print("🎉 ALL TASKS COMPLETED SUCCESSFULLY!")
print("="*70)

print("\n📊 COMPLETE SUMMARY:")
print("\n1. TRAINING RESUME:")
print("   ✓ Resumed from iteration: 160,000")
print("   ✓ Completed to iteration: 200,000")
print(f"   ✓ New checkpoints saved: {len(saved_models) if 'saved_models' in locals() else 'N/A'}")

print("\n2. MODEL ENSEMBLE:")
print("   ✓ Ensemble model created from top checkpoints")
print("   ✓ Model saved: generator_ensemble_final.pth")

print("\n3. COMPREHENSIVE EVALUATION:")
print("   ✓ 8 metrics evaluated (PSNR, SSIM, MS-SSIM, LPIPS, MAE, RMSE, NDVI, Edge)")
print("   ✓ 5 models compared (Nearest, Bilinear, Bicubic, SRCNN, Ours)")
print("   ✓ Statistical significance tests performed")

print("\n4. VISUALIZATIONS GENERATED:")
print("   ✓ Comprehensive evaluation charts (12 plots)")
print("   ✓ Visual quality comparison samples")
print("   ✓ Pixel-wise difference/error maps")
print("   ✓ Stability and convergence analysis")
print("   ✓ Failure case analysis")

print("\n5. FILES SAVED TO GOOGLE DRIVE:")
print("   📁 /content/drive/MyDrive/RFB-ESRGAN-Output/")
print("      • generator_ensemble_final.pth (Final model)")
print("      • generator_iter_170000.pth (Checkpoint)")
print("      • generator_iter_180000.pth (Checkpoint)")
print("      • generator_iter_190000.pth (Checkpoint)")
print("      • generator_iter_200000.pth (Checkpoint)")
print("      • comprehensive_evaluation.png")
print("      • visual_quality_samples.png")
print("      • difference_maps.png")
print("      • stability_analysis.png")
print("      • failure_cases.png")

print(f"\n🌐 View complete training logs: {wandb.run.url if wandb.run else 'N/A'}")

print("\n" + "="*70)
print("Thank you for using RFB-ESRGAN Training & Evaluation Pipeline!")
print("="*70)